# BGE-M3 Hybrid Retrieval Evaluation on BEIR

This notebook evaluates retrieval modes on a BEIR dataset and compares them:
1. Hybrid
2. HyDE (optional)
3. Re-Rank (optional)
4. All of the above


## 1. Colab setup

In [ ]:
USE_COLAB = False   # set True when running in Google Colab

if USE_COLAB:
    !git clone -b bryan/bgem3 https://github.com/KaiHackney/beir.git
    %cd beir
    !pip install -e .
    !pip install -U sentence-transformers datasets pytrec-eval-terrier huggingface_hub FlagEmbedding


## 2. Imports

In [ ]:
from __future__ import annotations

import logging
import os
import pathlib
import random

import numpy as np
import torch
from FlagEmbedding import BGEM3FlagModel
from tqdm.auto import tqdm

from beir import LoggingHandler, util
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.models.hyde import HyDE, HyDEPromptBuilder, HuggingFaceHypothesisGenerator
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES

logging.basicConfig(
    format="%(asctime)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
    handlers=[LoggingHandler()],
)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")


## 3. Configuration

Change `DATASET` to any BEIR dataset name

`DENSE_WEIGHT` and `SPARSE_WEIGHT` control hybrid fusion (BGE-M3 paper recommends `1.0` / `0.3`).

Set `USE_HYDE = True` to enable HyDE query expansion
Set `USE_RERANKER = True` to enable Reranking

In [ ]:
DATASET = "scifact"
BATCH_SIZE = 32
CORPUS_CHUNK_SIZE = 50000
TOP_K = 1000
DENSE_WEIGHT = 1.0
SPARSE_WEIGHT = 0.3
K_VALUES = [1, 3, 5, 10, 100, 1000]

# HyDE settings
USE_HYDE = False
HYDE_LOCAL_MODEL = "google/flan-t5-base"
HYDE_N_HYPOTHESES = 1
HYDE_MAX_NEW_TOKENS = 48
HYDE_DO_SAMPLE = False
HYDE_AGGREGATION = "max"
HYDE_INCLUDE_ORIGINAL_QUERY = True

# Reranking settings
USE_RERANKER = False
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
RERANKER_BATCH_SIZE = 32
RERANKER_TOP_K = 100

# Resolve paths based on environment
BASE_DIR  = "/content" if USE_COLAB else str(pathlib.Path(".").absolute())
DATA_DIR  = os.path.join(BASE_DIR, "datasets")
CACHE_DIR = os.path.join(BASE_DIR, "hyde_cache")


## 4. Download and load dataset

In [ ]:
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET}.zip"
data_path = util.download_and_unzip(url, DATA_DIR)

corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

print(f"Corpus size: {len(corpus):,}")
print(f"Queries:     {len(queries):,}")
print(f"Qrels:       {len(qrels):,}")

## 5. Load BGE-M3

In [ ]:
class BGEM3Adapter:
    """Wraps BGEM3FlagModel to implement the BEIR encode_queries/encode_corpus interface."""

    def __init__(self, model, batch_size=32):
        self.model = model
        self.batch_size = batch_size

    def _texts(self, corpus):
        if not corpus:
            return []
        if isinstance(corpus[0], str):
            return corpus
        return [f"{d.get('title', '')} {d.get('text', '')}".strip() for d in corpus]

    def encode_queries(self, queries, batch_size=None, **kwargs):
        out = self.model.encode(
            queries,
            batch_size=batch_size or self.batch_size,
            return_dense=True, return_sparse=False, return_colbert_vecs=False,
        )
        return torch.from_numpy(out["dense_vecs"])

    def encode_corpus(self, corpus, batch_size=None, **kwargs):
        texts = self._texts(corpus)
        out = self.model.encode(
            texts,
            batch_size=batch_size or self.batch_size,
            return_dense=True, return_sparse=False, return_colbert_vecs=False,
        )
        return torch.from_numpy(out["dense_vecs"])


bgem3 = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=device == "cuda",
    device=device,
)
adapter = BGEM3Adapter(bgem3, batch_size=BATCH_SIZE)
print("Model loaded.")


## 6. Encode corpus (dense + sparse)

In [ ]:
doc_ids = list(corpus.keys())
doc_list = [corpus[d] for d in doc_ids]          # list of {title, text} dicts
doc_texts = [f"{d.get('title', '')} {d.get('text', '')}".strip() for d in doc_list]

print(f"Encoding {len(doc_texts):,} documents (dense + sparse)...")
corpus_output = bgem3.encode(
    doc_texts,
    batch_size=BATCH_SIZE,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=False,
)
corpus_dense  = corpus_output["dense_vecs"]       # (num_docs, 1024)  numpy
corpus_sparse = corpus_output["lexical_weights"]  # list of {token_id: weight}
print(f"Done. Dense shape: {corpus_dense.shape}")


## 7. Encode queries (with optional HyDE expansion)

In [ ]:
query_ids   = list(queries.keys())
query_texts = [queries[q] for q in query_ids]

if USE_HYDE:
    print(f"HyDE enabled — loading generator: {HYDE_LOCAL_MODEL}")
    _gen_device = "cuda" if device == "cuda" else "cpu"
    generator = HuggingFaceHypothesisGenerator(
        model_name=HYDE_LOCAL_MODEL,
        n=HYDE_N_HYPOTHESES,
        max_new_tokens=HYDE_MAX_NEW_TOKENS,
        temperature=0.7,
        top_p=0.9,
        do_sample=HYDE_DO_SAMPLE,
        device=_gen_device,
    )
    dense_model = HyDE(
        base_model=adapter,
        generator=generator,
        prompt_builder=HyDEPromptBuilder(dataset=DATASET),
        cache_path=os.path.join(CACHE_DIR, f"{DATASET}.{HYDE_LOCAL_MODEL.replace('/', '_')}.jsonl"),
        include_original_query=HYDE_INCLUDE_ORIGINAL_QUERY,
        hypothesis_encoder="corpus",
        aggregation=HYDE_AGGREGATION,
    )
else:
    dense_model = adapter

dense_retriever = EvaluateRetrieval(
    DRES(dense_model, batch_size=BATCH_SIZE, corpus_chunk_size=CORPUS_CHUNK_SIZE),
    score_function="dot",
    k_values=K_VALUES,
)
dense_results = dense_retriever.retrieve(corpus, queries)
print("Dense retrieval complete.")

print("Encoding queries (sparse)...")
query_sparse_out = bgem3.encode(
    query_texts,
    batch_size=BATCH_SIZE,
    return_dense=False,
    return_sparse=True,
    return_colbert_vecs=False,
)
query_sparse = query_sparse_out["lexical_weights"]
print("Sparse query encoding complete.")


## 8. Sparse retrieval

In [ ]:
def sparse_score(query_weights, doc_weights):
    return sum(w * doc_weights[t] for t, w in query_weights.items() if t in doc_weights)

def min_max_normalize(matrix: np.ndarray) -> np.ndarray:
    mins = matrix.min(axis=1, keepdims=True)
    maxs = matrix.max(axis=1, keepdims=True)
    return (matrix - mins) / (maxs - mins + 1e-9)

print("Running sparse retrieval...")
sparse_score_matrix = np.zeros((len(query_ids), len(doc_ids)), dtype=np.float32)
for i, qw in enumerate(tqdm(query_sparse, desc="Sparse scoring")):
    for j, dw in enumerate(corpus_sparse):
        sparse_score_matrix[i, j] = sparse_score(qw, dw)

# Build sparse results dict (top-k per query)
sparse_results = {}
for i, qid in enumerate(query_ids):
    scores = sparse_score_matrix[i]
    top_idx = np.argpartition(scores, -TOP_K)[-TOP_K:]
    sparse_results[qid] = {doc_ids[j]: float(scores[j]) for j in top_idx}
print("Done.")


## 9. Hybrid retrieval (weighted-sum fusion)

In [ ]:
# Reconstruct dense score matrix from DRES results for fusion
print("Building hybrid score matrix...")
dense_score_matrix = np.zeros((len(query_ids), len(doc_ids)), dtype=np.float32)
doc_id_to_idx = {did: j for j, did in enumerate(doc_ids)}
for i, qid in enumerate(query_ids):
    for did, score in dense_results.get(qid, {}).items():
        j = doc_id_to_idx.get(did)
        if j is not None:
            dense_score_matrix[i, j] = score

hybrid_score_matrix = (
    DENSE_WEIGHT  * min_max_normalize(dense_score_matrix)
    + SPARSE_WEIGHT * min_max_normalize(sparse_score_matrix)
)

hybrid_results = {}
for i, qid in enumerate(query_ids):
    scores = hybrid_score_matrix[i]
    top_idx = np.argpartition(scores, -TOP_K)[-TOP_K:]
    hybrid_results[qid] = {doc_ids[j]: float(scores[j]) for j in top_idx}
print("Hybrid retrieval complete.")


## 10. Reranking (optional)

In [ ]:
if USE_RERANKER:
    from beir.reranking.models import CrossEncoder
    from beir.reranking import Rerank

    print(f"Loading reranker: {RERANKER_MODEL}")
    reranker = Rerank(CrossEncoder(RERANKER_MODEL), batch_size=RERANKER_BATCH_SIZE)

    print(f"Reranking top-{RERANKER_TOP_K} candidates per query...")
    reranked_results = reranker.rerank(corpus, queries, hybrid_results, top_k=RERANKER_TOP_K)
    print("Reranking complete.")
else:
    reranked_results = None
    print("Reranker disabled — set USE_RERANKER = True in config to enable.")


## 11. Evaluate

In [ ]:
evaluator = EvaluateRetrieval()
hyde_tag = " + HyDE" if USE_HYDE else ""

print("=" * 60)
print(f"Dataset: {DATASET}  |  HyDE: {USE_HYDE}  |  Reranker: {USE_RERANKER}")
print("=" * 60)

runs = [(f"Hybrid{hyde_tag}", hybrid_results)]
if reranked_results:
    runs.append((f"Hybrid{hyde_tag} + Reranker", reranked_results))

for name, results in runs:
    ndcg, map_, recall, precision = evaluator.evaluate(qrels, results, K_VALUES)
    mrr = evaluator.evaluate_custom(qrels, results, [10], metric="mrr")
    print(f"\n--- {name} ---")
    print(f"  NDCG@10:      {ndcg['NDCG@10']:.5f}")
    print(f"  Recall@100:   {recall['Recall@100']:.5f}")
    print(f"  Recall@1000:  {recall['Recall@1000']:.5f}")
    print(f"  MRR@10:       {mrr['MRR@10']:.5f}")


## 12. Inspect top-k results for a random query

In [ ]:
query_id = random.choice(query_ids)
print(f"Query: {queries[query_id]}\n")

hyde_tag = " + HyDE" if USE_HYDE else ""
runs = [(f"Hybrid{hyde_tag}", hybrid_results)]
if reranked_results:
    runs.append((f"Hybrid{hyde_tag} + Reranker", reranked_results))

for name, results in runs:
    print(f"--- Top 5 ({name}) ---")
    ranked = sorted(results[query_id].items(), key=lambda x: x[1], reverse=True)[:5]
    for rank, (doc_id, score) in enumerate(ranked, 1):
        title = corpus[doc_id].get("title", "")[:80]
        relevant = "*" if doc_id in qrels.get(query_id, {}) else " "
        print(f"  {relevant} {rank}. [{score:.4f}] {title}")
    print()
